# mp-spawn-workers — faded example 2: Launch workers with join=False and retrieve process IDs

> Practice drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `mp-spawn-workers`. The last cell reports your progress on the `Distributed: mp.spawn workers` subtopic back to Delta Drills.

**Most of the code is already written — complete the one blanked step**, run the test to check it, then run the last cell to record your progress.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Distributed: mp.spawn workers` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`mp-spawn-workers`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "mp-spawn-workers"
DD_SUBTOPIC = "Distributed: mp.spawn workers"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

`mp.spawn(..., join=False)` returns a `ProcessContext` object without blocking. The `ProcessContext.pids()` method returns the list of child process IDs, which can be used for monitoring or sending signals before calling `pc.join()` to wait for completion. This non-blocking form is essential when the launcher needs to do work while workers are running.

## Faded exercise 2

Implement `launch_nonblocking(spawn_module, worker_fn, world_size, port)` that:
1. Calls `spawn_module.spawn(worker_fn, args=(world_size, port), nprocs=world_size, join=False)`.
2. Returns the resulting `ProcessContext`-like object WITHOUT calling `.join()` on it.

Your task: **fill in the spawn call with `join=False` and return the context**.

**Your task:** complete the one blanked step in the code cell below. The surrounding code, function signatures, and variable names are given — work out the missing expression yourself, then run the test.

In [ ]:
def launch_nonblocking(spawn_module, worker_fn, world_size: int, port: int):
    raise NotImplementedError()  # TODO: fill in this step — read the prompt cell above

def _test():
    class _FakeCtx:
        def __init__(self, nprocs):
            self._pids = list(range(1000, 1000 + nprocs))
        def pids(self):
            return self._pids
        def join(self, timeout=None):
            return True

    class _FakeSpawn:
        def __init__(self):
            self.last_call = {}
        def spawn(self, fn, args, nprocs, join):
            self.last_call = {'fn': fn, 'args': args, 'nprocs': nprocs, 'join': join}
            return _FakeCtx(nprocs)

    fake_mod = _FakeSpawn()
    dummy_fn = lambda rank, ws, port: None
    pc = launch_nonblocking(fake_mod, dummy_fn, world_size=4, port=29503)
    assert fake_mod.last_call['join'] == False, "join should be False"
    assert fake_mod.last_call['nprocs'] == 4
    assert len(pc.pids()) == 4


def _test():
    class _FakeCtx:
        def __init__(self, nprocs):
            self._pids = list(range(2000, 2000 + nprocs))
        def pids(self): return self._pids
        def join(self, timeout=None): return True

    class _FakeSpawn:
        def __init__(self): self.calls = []
        def spawn(self, fn, args, nprocs, join):
            self.calls.append({'fn': fn, 'args': args, 'nprocs': nprocs, 'join': join})
            return _FakeCtx(nprocs)

    fake_mod = _FakeSpawn()
    fn = lambda rank, ws, port: None
    # world_size=2
    pc = launch_nonblocking(fake_mod, fn, world_size=2, port=29600)
    assert len(fake_mod.calls) == 1
    assert fake_mod.calls[0]['join'] == False, "must pass join=False"
    assert fake_mod.calls[0]['nprocs'] == 2
    assert fake_mod.calls[0]['args'] == (2, 29600)
    assert len(pc.pids()) == 2
    # world_size=3
    pc3 = launch_nonblocking(fake_mod, fn, world_size=3, port=29601)
    assert fake_mod.calls[1]['nprocs'] == 3
    assert len(pc3.pids()) == 3


try:
    _test()
    _dd_passed.add('faded2')
    print('[Delta Drills] faded2 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report your progress

Run the cell below to send your progress to Delta Drills. It only counts if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
def launch_nonblocking(spawn_module, worker_fn, world_size: int, port: int):
    pc = spawn_module.spawn(worker_fn, args=(world_size, port), nprocs=world_size, join=False)
    return pc

def _test():
    class _FakeCtx:
        def __init__(self, nprocs):
            self._pids = list(range(1000, 1000 + nprocs))
        def pids(self):
            return self._pids
        def join(self, timeout=None):
            return True

    class _FakeSpawn:
        def __init__(self):
            self.last_call = {}
        def spawn(self, fn, args, nprocs, join):
            self.last_call = {'fn': fn, 'args': args, 'nprocs': nprocs, 'join': join}
            return _FakeCtx(nprocs)

    fake_mod = _FakeSpawn()
    dummy_fn = lambda rank, ws, port: None
    pc = launch_nonblocking(fake_mod, dummy_fn, world_size=4, port=29503)
    assert fake_mod.last_call['join'] == False
    assert fake_mod.last_call['nprocs'] == 4
    assert len(pc.pids()) == 4
```
</details>